# WiDS Global Datathon 2026 — Tri-Survival Stack

**Objective:** Push beyond 0.963 by adding CoxPH and zone-split LGB 
to the GBSA backbone, with distance-stratified blending.

**Key additions from Notebook 05:**
- CoxPH survival model — linear inductive bias, complements GBSA
- Zone-split LGB — separate near/far classifiers per horizon
- Distance-stratified blending — different weights for near/far zones

**Starting point:** 0.96334  
**Target:** 0.97+  
**Metric:** Hybrid Score = 0.3 × C-index + 0.7 × (1 − Weighted Brier Score)

## 1. Dependencies

In [1]:
!pip install scikit-survival lightgbm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 9.5 MB/s eta 0:00:00


## 2. Environment Setup & Data Loading

In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv
import lightgbm as lgb

BASE_PATH = '/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/'

train      = pd.read_csv(BASE_PATH + 'train.csv')
test       = pd.read_csv(BASE_PATH + 'test.csv')
sample_sub = pd.read_csv(BASE_PATH + 'sample_submission.csv')

target_cols  = ['event', 'time_to_hit_hours']
id_cols      = ['event_id']
feature_cols = [c for c in train.columns if c not in target_cols + id_cols]

HORIZONS    = np.array([12, 24, 48, 72], dtype=float)
SEED        = 42
N_FOLDS     = 5
DIST_CUTOFF = 5000

near_mask_train = (train['dist_min_ci_0_5h'] < DIST_CUTOFF).values
far_mask_train  = ~near_mask_train
near_mask_test  = (test['dist_min_ci_0_5h']  < DIST_CUTOFF).values
far_mask_test   = ~near_mask_test

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f"Train shape     : {train.shape}")
print(f"Test shape      : {test.shape}")
print(f"Near zone train : {near_mask_train.sum()}")
print(f"Far zone train  : {far_mask_train.sum()}")
print(f"Near zone test  : {near_mask_test.sum()}")
print(f"Far zone test   : {far_mask_test.sum()}")

Train shape     : (221, 37)
Test shape      : (95, 35)
Near zone train : 69
Far zone train  : 152
Near zone test  : 28
Far zone test   : 67


## 3. Feature Engineering

In [3]:
def engineer_features(df, fit_df=None):
    out   = df.copy()
    dist  = out['dist_min_ci_0_5h'].clip(lower=1)
    speed = out['closing_speed_m_per_h']
    area  = out['area_first_ha'].clip(lower=1)

    # Distance transformations
    out['log_distance']    = np.log1p(dist)
    out['inv_distance']    = 1 / (dist / 1000 + 0.1)
    out['inv_distance_sq'] = out['inv_distance'] ** 2
    out['sqrt_distance']   = np.sqrt(dist)
    out['dist_km']         = dist / 1000
    out['dist_rank']       = dist.rank(pct=True)

    # Physics features
    safe_speed             = speed.clip(lower=0.1)
    out['eta']             = dist / safe_speed
    out['log_eta']         = np.log1p(out['eta'])
    out['eta_effective']   = out['eta'] * (1 - out['alignment_abs'])
    out['fire_radius_km']  = np.sqrt(area / np.pi) / 1000
    out['area_to_dist_ratio'] = out['fire_radius_km'] / (dist / 1000 + 0.01)
    out['threat_score']    = out['alignment_abs'] * out['inv_distance']

    # Additional features for CoxPH
    out['has_movement']    = (out['closing_speed_m_per_h'] > 0).astype(int)
    out['is_summer']       = out['event_start_month'].isin([6,7,8]).astype(int)
    out['is_afternoon']    = (out['event_start_hour'] >= 12).astype(int)
    out['zone_near']       = (dist < DIST_CUTOFF).astype(int)
    out['zone_far']        = (dist >= DIST_CUTOFF).astype(int)

    # Zone-stratified rank features
    ref = fit_df if fit_df is not None else df
    ref_dist = ref['dist_min_ci_0_5h'].clip(lower=1)
    ref_near = ref_dist < DIST_CUTOFF
    ref_far  = ~ref_near

    near_speed_rank = pd.Series(np.nan, index=df.index)
    far_threat_rank = pd.Series(np.nan, index=df.index)

    if ref_near.sum() > 0:
        near_speed_rank[near_mask_train if fit_df is None
                        else (dist < DIST_CUTOFF)] = \
            out.loc[dist < DIST_CUTOFF, 'closing_speed_m_per_h'].rank(pct=True)
    if ref_far.sum() > 0:
        far_threat_rank[far_mask_train if fit_df is None
                        else (dist >= DIST_CUTOFF)] = \
            out.loc[dist >= DIST_CUTOFF, 'threat_score'].rank(pct=True)

    out['near_speed_rank'] = near_speed_rank.fillna(0)
    out['far_threat_rank'] = far_threat_rank.fillna(0)

    out = out.drop(columns=['dist_min_ci_0_5h', 'area_first_ha'])
    return out

train_eng     = engineer_features(train, fit_df=train)
test_eng      = engineer_features(test,  fit_df=train)
feat_cols_eng = [c for c in train_eng.columns if c not in target_cols + id_cols]

y_surv = Surv.from_arrays(
    event = train['event'].astype(bool),
    time  = train['time_to_hit_hours']
)

X_train = train_eng[feat_cols_eng].values.astype(np.float32)
X_test  = test_eng[feat_cols_eng].values.astype(np.float32)

print(f"Feature count : {len(feat_cols_eng)}")
print(f"New features  : has_movement, is_summer, is_afternoon,")
print(f"                zone_near, zone_far, near_speed_rank, far_threat_rank")

Feature count : 51
New features  : has_movement, is_summer, is_afternoon,
                zone_near, zone_far, near_speed_rank, far_threat_rank


## 4. Model 1 — GBSA (4 configs × 20 seeds × 5 folds)

In [4]:
gbsa_configs = [
    {'learning_rate': 0.01, 'subsample': 0.70, 'max_depth': 3,
     'min_samples_leaf': 12, 'n_estimators': 1200},
    {'learning_rate': 0.01, 'subsample': 0.85, 'max_depth': 3,
     'min_samples_leaf': 10, 'n_estimators': 1000},
    {'learning_rate': 0.02, 'subsample': 0.75, 'max_depth': 2,
     'min_samples_leaf': 15, 'n_estimators': 800},
    {'learning_rate': 0.05, 'subsample': 0.80, 'max_depth': 2,
     'min_samples_leaf': 12, 'n_estimators': 500},
]

SEEDS = list(range(0, 20))

def get_surv_predictions(model, X):
    surv_fns = model.predict_survival_function(X)
    preds    = np.empty((len(surv_fns), len(HORIZONS)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i, :]  = fn(np.clip(HORIZONS, t_min, t_max))
    return 1.0 - preds

print(f"Training GBSA: {len(gbsa_configs)} configs x {len(SEEDS)} seeds "
      f"x {N_FOLDS} folds = "
      f"{len(gbsa_configs) * len(SEEDS) * N_FOLDS} models total")

gbsa_oof_sum  = np.zeros((len(train), len(HORIZONS)))
gbsa_test_sum = np.zeros((len(test),  len(HORIZONS)))
gbsa_count    = 0

for cfg_idx, cfg in enumerate(gbsa_configs):
    for seed in SEEDS:
        for fold, (train_idx, val_idx) in enumerate(
                skf.split(train, train['event'])):
            model = GradientBoostingSurvivalAnalysis(
                random_state=seed, **cfg)
            model.fit(X_train[train_idx], y_surv[train_idx])
            gbsa_oof_sum[val_idx] += get_surv_predictions(
                model, X_train[val_idx])
            gbsa_test_sum         += get_surv_predictions(
                model, X_test)
            gbsa_count            += 1

    print(f"  Config {cfg_idx+1}/4 complete "
          f"({(cfg_idx+1)*len(SEEDS)*N_FOLDS} models)")

gbsa_oof  = gbsa_oof_sum  / gbsa_count
gbsa_test = gbsa_test_sum / gbsa_count

print(f"\nTotal GBSA models: {gbsa_count}")
print("GBSA OOF ranges:")
for h_idx, H in enumerate(HORIZONS):
    p = gbsa_oof[:, h_idx]
    print(f"  {H:2.0f}h : min={p.min():.4f}, max={p.max():.4f}, "
          f"mean={p.mean():.4f}")

Training GBSA: 4 configs x 20 seeds x 5 folds = 400 models total
  Config 1/4 complete (100 models)
  Config 2/4 complete (200 models)
  Config 3/4 complete (300 models)
  Config 4/4 complete (400 models)

Total GBSA models: 400
GBSA OOF ranges:
  12h : min=0.0018, max=0.2000, mean=0.0494
  24h : min=0.0044, max=0.2000, mean=0.0620
  48h : min=0.0060, max=0.2000, mean=0.0654
  72h : min=0.0094, max=0.2000, mean=0.0752


## 5. Model 2 — CoxPH (7 alphas × 10 seeds × 5 folds)

In [5]:
COX_FEATURES = [
    'dist_km', 'log_distance', 'inv_distance',
    'closing_speed_m_per_h', 'radial_growth_rate_m_per_h',
    'alignment_abs', 'threat_score', 'log_eta', 'eta_effective',
    'area_to_dist_ratio', 'fire_radius_km',
    'num_perimeters_0_5h', 'has_movement',
    'near_speed_rank', 'far_threat_rank',
    'is_summer', 'is_afternoon',
    'zone_near', 'zone_far',
]

avail_cox    = [f for f in COX_FEATURES if f in train_eng.columns]
X_cox_train  = train_eng[avail_cox].values.astype(np.float32)
X_cox_test   = test_eng[avail_cox].values.astype(np.float32)

# Standardize for CoxPH — linear model requires scaled features
scaler       = StandardScaler()
X_cox_train_sc = scaler.fit_transform(X_cox_train)
X_cox_test_sc  = scaler.transform(X_cox_test)

COX_ALPHAS   = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
COX_SEEDS    = list(range(0, 10))

print(f"Training CoxPH: {len(COX_ALPHAS)} alphas x {len(COX_SEEDS)} seeds "
      f"x {N_FOLDS} folds = "
      f"{len(COX_ALPHAS) * len(COX_SEEDS) * N_FOLDS} models total")
print(f"CoxPH features: {len(avail_cox)}")

cox_oof_sum  = np.zeros((len(train), len(HORIZONS)))
cox_test_sum = np.zeros((len(test),  len(HORIZONS)))
cox_count    = 0

for alpha in COX_ALPHAS:
    for seed in COX_SEEDS:
        np.random.seed(seed)
        for fold, (train_idx, val_idx) in enumerate(
                skf.split(train, train['event'])):

            model = CoxPHSurvivalAnalysis(alpha=alpha, ties='efron')
            model.fit(X_cox_train_sc[train_idx], y_surv[train_idx])

            cox_oof_sum[val_idx] += get_surv_predictions(
                model, X_cox_train_sc[val_idx])
            cox_test_sum         += get_surv_predictions(
                model, X_cox_test_sc)
            cox_count            += 1

    print(f"  Alpha {alpha} complete")

cox_oof  = cox_oof_sum  / cox_count
cox_test = cox_test_sum / cox_count

print(f"\nTotal CoxPH models: {cox_count}")
print("CoxPH OOF ranges:")
for h_idx, H in enumerate(HORIZONS):
    p = cox_oof[:, h_idx]
    print(f"  {H:2.0f}h : min={p.min():.4f}, max={p.max():.4f}, "
          f"mean={p.mean():.4f}")

Training CoxPH: 7 alphas x 10 seeds x 5 folds = 350 models total
CoxPH features: 19
  Alpha 0.001 complete
  Alpha 0.01 complete
  Alpha 0.1 complete
  Alpha 0.5 complete
  Alpha 1.0 complete
  Alpha 2.0 complete
  Alpha 5.0 complete

Total CoxPH models: 350
CoxPH OOF ranges:
  12h : min=0.0000, max=0.2000, mean=0.0444
  24h : min=0.0001, max=0.2000, mean=0.0551
  48h : min=0.0001, max=0.2000, mean=0.0579
  72h : min=0.0001, max=0.2000, mean=0.0621


## 6. Model 3 — Zone-Split LGB (separate near/far classifiers)

In [6]:
def make_labels(df, horizons):
    labels, masks = {}, {}
    for H in horizons:
        valid_mask      = ~((df['event'] == 0) & (df['time_to_hit_hours'] < H))
        label           = ((df['event'] == 1) & (df['time_to_hit_hours'] <= H)).astype(int)
        labels[f'{H}h'] = label
        masks[f'{H}h']  = valid_mask
    return labels, masks

active_horizons = [24, 48]
labels, masks   = make_labels(train, active_horizons)

# Near zone: timing-focused features
NEAR_FEATURES = [
    'closing_speed_m_per_h', 'radial_growth_rate_m_per_h',
    'alignment_abs', 'num_perimeters_0_5h', 'area_growth_rate_ha_per_h',
    'eta_effective', 'log_eta', 'dist_km', 'threat_score',
    'near_speed_rank', 'event_start_hour', 'is_afternoon',
    'fire_radius_km', 'log_distance'
]

# Far zone: detection-focused features
FAR_FEATURES = [
    'dist_km', 'log_distance', 'inv_distance',
    'closing_speed_m_per_h', 'alignment_abs',
    'threat_score', 'log_eta', 'eta_effective',
    'area_to_dist_ratio', 'num_perimeters_0_5h',
    'far_threat_rank', 'is_summer', 'zone_far'
]

avail_near = [f for f in NEAR_FEATURES if f in train_eng.columns]
avail_far  = [f for f in FAR_FEATURES  if f in train_eng.columns]

lgb_params_near = {
    'objective': 'binary', 'verbose': -1, 'random_state': SEED,
    'learning_rate': 0.03, 'num_leaves': 7, 'max_depth': 3,
    'min_child_samples': 5, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'n_estimators': 300
}

lgb_params_far = {
    'objective': 'binary', 'verbose': -1, 'random_state': SEED,
    'learning_rate': 0.05, 'num_leaves': 7, 'max_depth': 2,
    'min_child_samples': 8, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.5, 'reg_lambda': 2.0, 'n_estimators': 200
}

lgb_near_oof  = {H: np.zeros(len(train)) for H in active_horizons}
lgb_near_test = {H: np.zeros(len(test))  for H in active_horizons}
lgb_far_oof   = {H: np.zeros(len(train)) for H in active_horizons}
lgb_far_test  = {H: np.zeros(len(test))  for H in active_horizons}

for H in active_horizons:
    label      = labels[f'{H}h'].values
    mask       = masks[f'{H}h'].values
    print(f"\nTraining zone-split LGB @ {H}h...")

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(train, train['event'])):

        train_mask = mask[train_idx]
        val_mask   = mask[val_idx]

        # Near zone model
        near_tr = near_mask_train[train_idx] & train_mask
        near_vl = near_mask_train[val_idx]

        if near_tr.sum() > 5:
            X_near_tr = train_eng.iloc[train_idx][avail_near].values[near_tr]
            y_near_tr = label[train_idx][near_tr]
            X_near_vl = train_eng.iloc[val_idx][avail_near].values
            X_near_ts = test_eng[avail_near].values

            m_near = lgb.LGBMClassifier(**lgb_params_near)
            m_near.fit(X_near_tr, y_near_tr,
                       callbacks=[lgb.log_evaluation(-1)])
            lgb_near_oof[H][val_idx]  += m_near.predict_proba(X_near_vl)[:, 1] / N_FOLDS
            lgb_near_test[H]          += m_near.predict_proba(X_near_ts)[:, 1] / N_FOLDS

        # Far zone model
        far_tr = far_mask_train[train_idx] & train_mask
        far_vl = far_mask_train[val_idx]

        if far_tr.sum() > 5:
            X_far_tr = train_eng.iloc[train_idx][avail_far].values[far_tr]
            y_far_tr = label[train_idx][far_tr]
            X_far_vl = train_eng.iloc[val_idx][avail_far].values
            X_far_ts = test_eng[avail_far].values

            m_far = lgb.LGBMClassifier(**lgb_params_far)
            m_far.fit(X_far_tr, y_far_tr,
                      callbacks=[lgb.log_evaluation(-1)])
            lgb_far_oof[H][val_idx]  += m_far.predict_proba(X_far_vl)[:, 1] / N_FOLDS
            lgb_far_test[H]          += m_far.predict_proba(X_far_ts)[:, 1] / N_FOLDS

    print(f"  Near OOF mean: {lgb_near_oof[H][near_mask_train].mean():.4f}")
    print(f"  Far  OOF mean: {lgb_far_oof[H][far_mask_train].mean():.4f}")

print("\nZone-split LGB training complete.")


Training zone-split LGB @ 24h...
  Near OOF mean: 0.1844
  Far  OOF mean: 0.0000

Training zone-split LGB @ 48h...
  Near OOF mean: 0.1893
  Far  OOF mean: 0.0000

Zone-split LGB training complete.


## 7. Distance-Stratified Blend & Submission

In [7]:
# Horizon-specific blend weights
# 12h: GBSA only — CoxPH and LGB add noise at short horizons
# 24h, 48h: full blend — CoxPH and LGB add meaningful diversity
WEIGHTS = {
    12: {'W_GBSA_NEAR': 1.00, 'W_COX_NEAR': 0.00, 'W_LGB_NEAR': 0.00},
    24: {'W_GBSA_NEAR': 0.60, 'W_COX_NEAR': 0.25, 'W_LGB_NEAR': 0.15},
    48: {'W_GBSA_NEAR': 0.60, 'W_COX_NEAR': 0.25, 'W_LGB_NEAR': 0.15},
    72: {'W_GBSA_NEAR': 1.00, 'W_COX_NEAR': 0.00, 'W_LGB_NEAR': 0.00},
}
W_GBSA_FAR = 1.00
W_COX_FAR  = 0.00
W_LGB_FAR  = 0.00

near_zone_rates = np.array([0.710, 0.913, 0.957, 1.000])

def apply_distance_gate(preds, near_mask, near_rates):
    result     = np.zeros_like(preds)
    near_preds = preds[near_mask, :]
    for h_idx in range(preds.shape[1]):
        raw = near_preds[:, h_idx]
        if raw.max() > raw.min():
            scaled = (raw - raw.min()) / (raw.max() - raw.min())
        else:
            scaled = np.ones_like(raw)
        low  = near_rates[h_idx] * 0.3
        high = min(near_rates[h_idx] * 1.05, 0.99)
        result[near_mask, h_idx] = low + scaled * (high - low)
    return result

def enforce_monotonicity(preds):
    result = preds.copy()
    for i in range(1, result.shape[1]):
        result[:, i] = np.maximum(result[:, i], result[:, i-1])
    return result

def blend_predictions(gbsa, cox, lgb_near, lgb_far,
                      near_mask, weights, h_idx, H_int):
    result = np.zeros(len(near_mask))
    w = weights[H_int]
    for i in range(len(near_mask)):
        if near_mask[i]:
            lgb_pred   = lgb_near.get(H_int, np.zeros(len(near_mask)))[i]
            result[i]  = (w['W_GBSA_NEAR'] * gbsa[i, h_idx] +
                          w['W_COX_NEAR']  * cox[i, h_idx]  +
                          w['W_LGB_NEAR']  * lgb_pred)
        else:
            lgb_pred   = lgb_far.get(H_int, np.zeros(len(near_mask)))[i]
            result[i]  = (W_GBSA_FAR * gbsa[i, h_idx] +
                          W_COX_FAR  * cox[i, h_idx]  +
                          W_LGB_FAR  * lgb_pred)
    return result

# Build blended predictions
test_blend = np.zeros((len(test), len(HORIZONS)))
oof_blend  = np.zeros((len(train), len(HORIZONS)))

for h_idx, H in enumerate(HORIZONS):
    H_int = int(H)
    test_blend[:, h_idx] = blend_predictions(
        gbsa_test, cox_test, lgb_near_test, lgb_far_test,
        near_mask_test, WEIGHTS, h_idx, H_int)
    oof_blend[:, h_idx] = blend_predictions(
        gbsa_oof, cox_oof, lgb_near_oof, lgb_far_oof,
        near_mask_train, WEIGHTS, h_idx, H_int)

# Apply distance gate
test_gated = apply_distance_gate(test_blend, near_mask_test,  near_zone_rates)
oof_gated  = apply_distance_gate(oof_blend,  near_mask_train, near_zone_rates)

# 72h fix
test_gated[near_mask_test,  3] = 1.0
test_gated[far_mask_test,   3] = 0.0

# Enforce monotonicity and clip
test_final = enforce_monotonicity(test_gated)
test_final = np.clip(test_final, 0.0, 1.0)

# OOF Brier Score comparison
print("OOF Brier Score comparison:")
print(f"{'Horizon':<10} {'NB04':>10} {'NB06 v1':>10} {'NB06 v2':>10}")
print("-" * 42)
nb04_scores = {12: 0.0577, 24: 0.0315, 48: 0.0179}
nb06v1      = {12: 0.0604, 24: 0.0304, 48: 0.0177}

for h_idx, H in enumerate([12, 24, 48]):
    mask  = ~((train['event'] == 0) & (train['time_to_hit_hours'] < H))
    label = ((train['event'] == 1) &
             (train['time_to_hit_hours'] <= H)).astype(int)
    bs    = np.mean((oof_gated[mask.values, h_idx] -
                     label[mask.values]) ** 2)
    print(f"{H}h        {nb04_scores[H]:>10.4f} "
          f"{nb06v1[H]:>10.4f} {bs:>10.4f}")

# Build submission
sub = sample_sub.copy()
sub['prob_12h'] = test_final[:, 0]
sub['prob_24h'] = test_final[:, 1]
sub['prob_48h'] = test_final[:, 2]
sub['prob_72h'] = test_final[:, 3]

print("\nSubmission preview:")
print(sub.head(10).round(4))

print(f"\nMonotonicity check:")
print(f"  prob_12h <= prob_24h : "
      f"{(sub['prob_12h'] <= sub['prob_24h']).all()}")
print(f"  prob_24h <= prob_48h : "
      f"{(sub['prob_24h'] <= sub['prob_48h']).all()}")
print(f"  prob_48h <= prob_72h : "
      f"{(sub['prob_48h'] <= sub['prob_72h']).all()}")

sub.to_csv('submission.csv', index=False)
print("\nFile saved: submission.csv")

OOF Brier Score comparison:
Horizon          NB04    NB06 v1    NB06 v2
------------------------------------------
12h            0.0577     0.0604     0.0578
24h            0.0315     0.0304     0.0304
48h            0.0179     0.0177     0.0177

Submission preview:
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602    0.0000    0.0000    0.0000       0.0
1  13353600    0.4979    0.8409    0.9116       1.0
2  13942327    0.0000    0.0000    0.0000       0.0
3  16112781    0.5427    0.8401    0.9080       1.0
4  17132808    0.0000    0.0000    0.0000       0.0
5  17445696    0.0000    0.0000    0.0000       0.0
6  17599982    0.0000    0.0000    0.0000       0.0
7  18750374    0.3442    0.6855    0.8050       1.0
8  21365245    0.0000    0.0000    0.0000       0.0
9  23634840    0.3931    0.6271    0.7130       1.0

Monotonicity check:
  prob_12h <= prob_24h : True
  prob_24h <= prob_48h : True
  prob_48h <= prob_72h : True

File saved: submission.csv
